# SHIELD OpenEMR Heathcare EHR Security 

Demonstrate SHIELD detecting cyberattacks on a real
OpenEMR Electronic Health Record system running in a simulated
NHS healtcare environment.

## This notebook works as following steps 
1. Loads normal and attack traffic captured from OpenEMR
2. Engineers SHIELD-compatible features from raw Wireshark packets
3. Uses Gradient Boosting with Stratified K-Fold cross-validation
4. Shows detection results with clinical interpretation
5. Produces figures suitable for the paper

## Explained below to Gradient Boosting instead of deep learning in this model 
The OpenEMR capture contains ~4,000 packets total. Deep learning
models require tens of thousands of samples to generalise reliably.
Gradient Boosting with cross-validation is the standard approach
for small network traffic datasets and produces honest, reproducible
results. This is consistent with SHIELD Layer 3 which uses the
best available detector for each data context.

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────
!pip install scikit-learn pandas numpy matplotlib seaborn -q

In [ ]:
# ── Cell 2: Imports ────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, f1_score, recall_score,
    accuracy_score, roc_curve
)

np.random.seed(42)
print('Libraries loaded.')

In [ ]:
# ── Cell 3: Load Wireshark captures ───────────────────────────
# Change paths if files are in a different location

NORMAL_PATH = '/content/wireshark.csv'
ATTACK_PATH = '/content/wireshark_attack.csv'

df_normal = pd.read_csv(NORMAL_PATH)
df_attack = pd.read_csv(ATTACK_PATH)

print(f'Normal traffic : {len(df_normal):,} packets')
print(f'Attack traffic : {len(df_attack):,} packets')
print(f'\nNormal protocols  : {df_normal["Protocol"].value_counts().to_dict()}')
print(f'Attack protocols  : {df_attack["Protocol"].value_counts().to_dict()}')

# Show attack evidence
attack_info = df_attack['Info'].fillna('')
print(f'\nAttack indicators confirmed:')
print(f'  Failed logins (error=1) : {attack_info.str.contains("error=1", na=False).sum()}')
print(f'  POST login attempts     : {attack_info.str.contains("POST", na=False).sum()}')
print(f'  Auth endpoints hit      : {attack_info.str.contains("auth=login", na=False).sum()}')

In [ ]:
# ── Cell 4: Feature Engineering ───────────────────────────────
# Convert raw Wireshark packet fields into SHIELD-compatible features.
# Features are engineered from Protocol, Length, Time, and Info fields.

def engineer_features(df, label):
    """
    Engineers SHIELD-compatible features from Wireshark packet data.

    Feature groups:
    - Protocol encoding  : TCP, HTTP, UDP flags
    - Packet length      : raw, log-transformed, size bands
    - Timing             : inter-packet delta, rapid burst detection
    - HTTP indicators    : login requests, POST submissions, error codes
    - Rolling windows    : cumulative error count over last 5 packets
    """
    feat = pd.DataFrame(index=df.index)

    # Protocol
    feat['is_tcp']       = (df['Protocol'] == 'TCP').astype(int)
    feat['is_http']      = (df['Protocol'] == 'HTTP').astype(int)
    feat['is_http_json'] = (df['Protocol'] == 'HTTP/JSON').astype(int)
    feat['is_udp']       = (df['Protocol'] == 'UDP').astype(int)
    feat['is_tls']       = df['Protocol'].str.contains('TLS', na=False).astype(int)

    # Packet length
    feat['pkt_len']     = pd.to_numeric(df['Length'], errors='coerce').fillna(0)
    feat['pkt_len_log'] = np.log1p(feat['pkt_len'])
    feat['is_large']    = (feat['pkt_len'] > 500).astype(int)
    feat['is_small']    = (feat['pkt_len'] < 60).astype(int)

    # Timing
    time_vals           = pd.to_numeric(df['Time'], errors='coerce').fillna(0)
    feat['time_delta']  = time_vals.diff().fillna(0).clip(lower=0)
    feat['td_log']      = np.log1p(feat['time_delta'])
    feat['is_rapid']    = (feat['time_delta'] < 0.1).astype(int)
    feat['is_very_rapid'] = (feat['time_delta'] < 0.01).astype(int)

    # HTTP indicators from Info field
    info = df['Info'].fillna('')
    feat['is_login_page']    = info.str.contains('login.php', case=False, na=False).astype(int)
    feat['is_post']          = info.str.contains('POST', na=False).astype(int)
    feat['is_login_error']   = info.str.contains('error=1|login_screen', case=False, na=False).astype(int)
    feat['is_auth_endpoint'] = info.str.contains('auth=login|main_screen', case=False, na=False).astype(int)
    feat['is_200_ok']        = info.str.contains('200 OK', na=False).astype(int)
    feat['is_302']           = info.str.contains('302', na=False).astype(int)
    feat['is_syn']           = info.str.contains('SYN', na=False).astype(int)
    feat['is_rst']           = info.str.contains('RST', na=False).astype(int)
    feat['is_fin']           = info.str.contains('FIN', na=False).astype(int)

    # Rolling window cumulative attack signals over last 5 packets
    # This is the key feature: brute force creates a burst of login errors
    feat['login_errors_5']  = feat['is_login_error'].rolling(5, min_periods=1).sum()
    feat['post_count_5']    = feat['is_post'].rolling(5, min_periods=1).sum()
    feat['rapid_count_5']   = feat['is_rapid'].rolling(5, min_periods=1).sum()
    feat['auth_count_5']    = feat['is_auth_endpoint'].rolling(5, min_periods=1).sum()

    # Rolling window over 10 packets — catches slower attacks
    feat['login_errors_10'] = feat['is_login_error'].rolling(10, min_periods=1).sum()
    feat['post_count_10']   = feat['is_post'].rolling(10, min_periods=1).sum()

    feat['label'] = label
    feat.fillna(0, inplace=True)
    return feat


print('Engineering features...')
feat_normal = engineer_features(df_normal, label=0)
feat_attack = engineer_features(df_attack, label=1)

# Combine
df_all = pd.concat([feat_normal, feat_attack], ignore_index=True)

feature_cols = [c for c in df_all.columns if c != 'label']

X = df_all[feature_cols].values.astype(np.float32)
y = df_all['label'].values.astype(int)

# Scale
scaler = StandardScaler()
X = scaler.fit_transform(X)

print(f'Total samples  : {len(X):,}')
print(f'Normal         : {(y==0).sum():,}')
print(f'Attack         : {(y==1).sum():,}')
print(f'Features       : {X.shape[1]}')

In [ ]:
#  Cell 5: Train with Stratified K-Fold Cross-Validation ─
# Stratified K-Fold is the correct approach for small datasets.
# It ensures each fold has the same class ratio as the full dataset
# and uses all data for both training and testing.
#
# Gradient Boosting is used because:
# - Works well on small tabular datasets (4,000 samples)
# - No overfitting risk that deep models have on small data
# - Produces reliable, reproducible results
# - Interpretable feature importances for the paper

clf = GradientBoostingClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    min_samples_leaf=5,
    random_state=42
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# cross_val_predict gives out-of-fold predictions for every sample
print('Running 5-fold cross-validation...')
y_prob = cross_val_predict(clf, X, y, cv=skf, method='predict_proba')[:, 1]
y_pred = cross_val_predict(clf, X, y, cv=skf, method='predict')

print('Done.')

In [ ]:
# ── Cell 6: Results ────────────────────────────────────────────

tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()
acc    = accuracy_score(y, y_pred)
f1     = f1_score(y, y_pred, zero_division=0)
recall = recall_score(y, y_pred, zero_division=0)
auc    = roc_auc_score(y, y_prob)

print('\n' + '='*58)
print('  SHIELD — OpenEMR Brute Force Attack Detection Results')
print('='*58)
print(f'  Accuracy  : {acc:.4f}')
print(f'  F1-Score  : {f1:.4f}')
print(f'  Recall    : {recall:.4f}  ← attack detection rate')
print(f'  AUC-ROC   : {auc:.4f}')
print(f'\n  Confusion Matrix:')
print(f'    True Positives  (attacks caught)  : {tp}')
print(f'    False Positives (false alarms)    : {fp}')
print(f'    True Negatives  (normal confirmed): {tn}')
print(f'    False Negatives (missed attacks)  : {fn}')
print('='*58)
print('\nClassification Report:')
print(classification_report(y, y_pred,
      target_names=['Normal', 'Attack (Brute Force)']))

print('Clinical interpretation:')
if recall >= 0.85:
    print(f'   SHIELD detects {recall*100:.1f}% of brute force login attacks')
    print(f'     on the OpenEMR Healthcare EHR system.')
elif recall >= 0.70:
    print(f'    SHIELD detects {recall*100:.1f}% of attacks - acceptable for examin.')
    print(f'     More training data would improve performance.')
else:
    print(f'  Detection rate {recall*100:.1f}% — review feature engineering.')

In [ ]:
# ── Cell 7: Feature Importance ─────────────────────────────────
# Train final model on all data to extract feature importances
# This shows WHICH features SHIELD uses to detect the attack

clf.fit(X, y)
importances = pd.Series(
    clf.feature_importances_,
    index=feature_cols
).sort_values(ascending=False)

print('Top 10 most important features for attack detection:')
print(importances.head(10).to_string())

In [ ]:
# ── Cell 8: Visualisations ─────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ── Plot 1: Confusion Matrix ───────────────────────────────────
cm = confusion_matrix(y, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Attack'],
            yticklabels=['Normal', 'Attack'],
            ax=axes[0], annot_kws={'size': 14})
axes[0].set_title('SHIELD Detection - OpenEMR\nBrute Force Login Attack',
                   fontweight='bold')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# ── Plot 2: ROC Curve ──────────────────────────────────────────
fpr, tpr, _ = roc_curve(y, y_prob)
axes[1].plot(fpr, tpr, color='steelblue', lw=2,
             label=f'SHIELD (AUC = {auc:.4f})')
axes[1].plot([0,1],[0,1], 'k--', lw=1, label='Random baseline')
axes[1].fill_between(fpr, tpr, alpha=0.1, color='steelblue')
axes[1].set_title('ROC Curve - OpenEMR ',
                   fontweight='bold')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate (Recall)')
axes[1].legend(loc='lower right')
axes[1].grid(True, alpha=0.3)

# ── Plot 3: Top Feature Importances ───────────────────────────
top10 = importances.head(10)
colors = ['#d32f2f' if 'login' in f or 'auth' in f or 'post' in f or 'error' in f
          else 'steelblue'
          for f in top10.index]
axes[2].barh(range(len(top10)), top10.values[::-1], color=colors[::-1])
axes[2].set_yticks(range(len(top10)))
axes[2].set_yticklabels(top10.index[::-1], fontsize=9)
axes[2].set_title('Top Features for Attack Detection\n(red = attack-specific)',
                   fontweight='bold')
axes[2].set_xlabel('Feature Importance')
axes[2].grid(axis='x', alpha=0.3)

plt.suptitle(
    'SHIELD - OpenEMR healthcare EHR Security \n'
    'Brute Force Login Attack Detection (Simulated NHS Environment)',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig('shield_openemr.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: shield_openemr.png')

# ── Cell 9: Save Results ───────────────────────────────────────

examin_results = pd.DataFrame([{
    'Dataset':     'OpenEMR healthcare EHR (Simulated NHS Environment)',
    'Method':      'Gradient Boosting + Stratified 5-Fold CV',
    'Normal_pkts': int((y==0).sum()),
    'Attack_pkts': int((y==1).sum()),
    'Accuracy':    round(acc, 4),
    'F1_Score':    round(f1, 4),
    'Recall':      round(recall, 4),
    'AUC_ROC':     round(auc, 4),
    'TP': int(tp), 'FP': int(fp),
    'TN': int(tn), 'FN': int(fn),
    'Attack_type': 'Brute Force Login (repeated failed authentication)'
}])

examin_results.to_csv('shield_openemr_results.csv', index=False)

# Save feature importances
importances.reset_index().rename(
    columns={'index': 'Feature', 0: 'Importance'}
).to_csv('shield_openemr_features.csv', index=False)

print('Saved: shield_openemr_results.csv')
print('Saved: shield_openemr_features.csv')
print('Saved: shield_openemr.png')
print('\n OpenEMR result complete.')
print('real-world healthcare environment validation.')

In [ ]:
# ── Save Results CSV ───────────────────────────────────────────
from sklearn.metrics import accuracy_score, recall_score, f1_score

examin_results = pd.DataFrame([{
    'Dataset':     'OpenEMR healthcare EHR (Simulated NHS Environment)',
    'Method':      'Gradient Boosting + Stratified 5-Fold CV',
    'Normal_pkts': int((y==0).sum()),
    'Attack_pkts': int((y==1).sum()),
    'Accuracy':    round(accuracy_score(y, y_pred), 4),
    'F1_Score':    round(f1_score(y, y_pred), 4),
    'Recall':      round(recall_score(y, y_pred), 4),
    'AUC_ROC':     round(auc, 4),
    'TP': int(tp), 'FP': int(fp),
    'TN': int(tn), 'FN': int(fn),
    'Attack_type': 'Brute Force Login (repeated failed authentication)'
}])

examin_results.to_csv('shield_openemr_results.csv', index=False)
importances.reset_index().rename(
    columns={'index':'Feature', 0:'Importance'}
).to_csv('shield_openemr_features.csv', index=False)

print('Saved: shield_openemr_results.csv')
print('Saved: shield_openemr_features.csv')
print('\n OpenEMR examin complete.')